# Plotting notebook for background


In [12]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
from helpers import merge_bkg, merge_cutflows
import shutil
from helpers import *

In [13]:
# configs for 32e Carefull with all this definitions: All work should be done here

vr = "32d"

bkgttj = ["TTJets"]
bkgdyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50",]
bkgqcd = ["QCD_Pt15To20", "QCD_Pt20To30", "QCD_Pt30To50", "QCD_Pt50To80", "QCD_Pt80To120", "QCD_Pt120To170", "QCD_Pt170To300", "QCD_Pt300To470", "QCD_Pt470To600",
           "QCD_Pt600To800", "QCD_Pt800To1000", "QCD_Pt1000"]

channels = ["bkg_base_2mulj",
            "bkg_base_2mulj_iso",
            "bkg_base_2mulj_iso_disp",
            "bkg_base_2mulj_iso_disp_dphi",
            "bkg_base_2mulj_iso_disp_dphi_mass150",
            "bkg_base_2mulj_iso_disp_dphi_mass175",
]

output = coffea.util.load(f"outputs/bkg_{vr}.coffea")

## Final yields summary

In [18]:
ch = channels[-1]#"bkg_base_2mulj_iso_disp_dphi_mass175"
signal = fmulxy4 + fmulxy5 + fmulxy6 + fmulxy7 + fmulxy8 + fmulxy9 + fmulxy0 + fmulxyi + fmulxyx
cut = "ljljMass > 175"
cuts = ["LJ-LJ dPhi > 2", "ljljMass > 150", "ljljMass > 175"]

print(f"{'Sample':<30} {'Yield':>12}")
#signal yield
for im, sample in enumerate(signal):
    if "500" in sample:
        outf = f"_fmu32d_{2}"
        print(outf)
    # print(f"{sample:<30}: {output[sample]['cutflow'][ch].rows[cut]['weighted']:>12.2f}")
    
# Background yields
dyj = merge_cutflows(output, bkgdyj, ch)
ttj = merge_cutflows(output, bkgttj, ch)
qcd = merge_cutflows(output, bkgqcd, ch)
   
print(f"{'TTJets':<30} {ttj[cut]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'DYJ':<30}    {dyj[cut]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'QCD':<30}    {qcd[cut]['weighted']:>12.2f}")

Sample                                Yield
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
_fmu32d_2
TTJets                               150.19
--------------------------------------------
DYJ                                     433.50
--------------------------------------------
QCD                                       0.80


## Number of events after every cut's been applied

In [ ]:
ch = chan[-1]
print(ch)
for i in signal:
    print(i)
    output[i]["cutflow"][ch].print_table()
    print()

In [ ]:
DYJ = merge_cutflows(output, bkgdyj, ch)
TTJ = merge_cutflows(output, bkgttj, ch)
QCD = merge_cutflows(output, bkgqcd, ch)

backgrounds = {
    "DYJ": bkgdyj,
    "TTJ": bkgttj,
    "QCD": bkgqcd,
}

merged_cf = {}

for name, samples in backgrounds.items():
    merged_cf[name] = merge_cutflows(output, samples, ch)

for name, cf in merged_cf.items():
    print(f"\n{name}")
    print(f"{'cut name':<20} {'raw':>12} {'weighted':>15}")
    print("-" * 50)

    for cut, vals in cf.items():
        print(
            f"{cut:<20}"
            f"{vals['raw']:>12,.0f}"
            f"{vals['weighted']:>15,.1f}"
        )

In [ ]:
all_bkgs = bkgdyj + bkgttj + bkgqcd
total_bkg = merge_cutflows(output, all_bkgs, ch)

for cut, vals in total_bkg.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

In [21]:
mxx_map = {"500GeV": "2", "800GeV": "3", "1000GeV": "4"}
mzd_map = {"0p25GeV": "1", "1p2GeV": "2", "5p0GeV": "3"}

groups = [
    fmulxy4,
    fmulxy5,
    fmulxy6,
    fmulxy7,
    fmulxy8,
    fmulxy9,
    fmulxy0,
    fmulxyi,
    fmulxyx,
]

with open("combine_inputs.txt", "w") as fout:

    for group in groups:

        for ctau_idx, sample in enumerate(group, start=1):

            yld = output[sample]['cutflow'][ch].rows[cut]['weighted']

            _, mxx, mzd, _ = sample.split("_")

            identifier = (
                mxx_map[mxx]
                + mzd_map[mzd]
                + str(ctau_idx)
            )

            fout.write(f"_fmu_32d_{identifier}v2 {yld:.2f}\n")